# Instrument control with PySerial

### A virtual digital multimeter on the other end of a serial port

Every instrument on a bench has the same shape of interface: you send it a line of
text, it sends a line of text back. Learning that shape is most of the job. This
notebook does it with **PySerial** and nothing else — no PyVISA, no driver class,
no helper functions. Every exchange is written out longhand so you can see exactly
what goes onto the wire.

## Before you run anything

Open a terminal in this folder and start the instrument:

```bash
uv run virtual_dmm.py
```

A front panel appears. Leave it open and put it beside this notebook — every
command you send will scroll past in its traffic log, colour-coded, with timings.
That log is half the point of the exercise.

## What `socket://` is doing

PySerial opens more than COM ports. `serial.serial_for_url()` understands a small
family of URL schemes, and `socket://host:port` makes a TCP connection look
exactly like a serial port to everything above it. The object you get back is a
real `serial.Serial`: real bytes, real timeouts, real line termination.

| | Real hardware | This notebook |
|---|---|---|
| URL | `COM4` or `/dev/ttyUSB0` | `socket://127.0.0.1:5025` |
| Everything below | `serial_for_url(...)` | `serial_for_url(...)` |
| Everything above | identical | identical |

The one thing the simulation cannot teach you is UART configuration. Baud rate,
parity, stop bits and flow control are all ignored by the socket handler — there
is no UART to configure. On real hardware they are the first thing to get wrong.

---

In [ ]:
# Initialization
from warnings import filterwarnings
filterwarnings('ignore')

# Plot in the same cell
%matplotlib inline

---
## 0 · Open the port

In [ ]:
import serial

PORT = "socket://127.0.0.1:5025"

# baudrate, bytesize, parity and stopbits are accepted and ignored by the
# socket:// handler.  They are written out here anyway, because on a real
# instrument they are the settings you will spend an afternoon getting wrong.
dmm = serial.serial_for_url(
    PORT,
    baudrate=9600,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_NONE,
    stopbits=serial.STOPBITS_ONE,
    timeout=5,          # seconds to wait for a reply before giving up
    write_timeout=5,
)

print(dmm)
print("open:", dmm.is_open)

### Two things that will bite you

**Termination.** The instrument reads until it sees a newline. Send `b"*IDN?"`
without the `\n` and it sits there holding your command, waiting for the rest of
the line, until your `timeout` expires and you get back `b""`. Try it deliberately
once — it is the single most common failure in serial instrument control, and
recognising the symptom saves you an hour later.

**Reading back.** `dmm.read(n)` needs to know how many bytes to expect, which you
usually don't. `dmm.read_until(b"\n")` reads until the terminator or the timeout,
whichever comes first. That is what we use throughout. Note that it returns
*everything including* the `\n`, so `.strip()` before parsing.

`dmm.readline()` also exists and does much the same thing, but `read_until` says
what it means and lets you pick the terminator.

---
## 1 · `*IDN?` — the hello world of instrument control

Every SCPI instrument ever made answers `*IDN?` with four comma-separated fields:
manufacturer, model, serial number, firmware revision. It is the first thing you
send to anything, because it proves the whole chain works — cable, port settings,
termination, parsing — before you have written a single line of measurement code.

In [ ]:
# Throw away anything left in the receive buffer from a previous run.  Without
# this, a reply you never read for an earlier command turns up as the answer to
# this one, and every reply after it is off by one.  Instrument code that
# mysteriously "lags by one command" is almost always missing this line.
dmm.reset_input_buffer()

dmm.write(b"*IDN?\n")

raw = dmm.read_until(b"\n")
print("raw bytes :", raw)

idn = raw.decode("ascii").strip()
print("decoded   :", idn)

maker, model, serial_number, firmware = idn.split(",")
print()
print(f"  manufacturer  {maker}")
print(f"  model         {model}")
print(f"  serial number {serial_number}")
print(f"  firmware      {firmware}")

assert model == "34461A", f"expected a 34461A, got {model!r}"

### Queries and commands are not the same thing

A SCPI header ending in `?` is a **query**: it produces a reply, and you must read
that reply. Anything else is a **command**: it produces nothing, and if you try to
read a reply you will block until the timeout.

Getting this wrong in either direction desynchronises the link. Read when there is
nothing to read and you stall; fail to read when there is something and it comes
back as the answer to your *next* query.

In [ ]:
import time

# A command.  No reply - so we do not read one.
dmm.write(b"*CLS\n")

# Proof: give the instrument a generous moment, then look at the buffer.
time.sleep(0.2)
print("bytes waiting after a command:", dmm.in_waiting)

# A query.  There is a reply, so we must take it.
dmm.write(b"SYST:ERR?\n")
print("error queue:", dmm.read_until(b"\n").decode().strip())

---
## 2 · Configure DC volts and take one reading

`CONFigure` sets up a measurement without performing it. `READ?` performs it and
returns the result.

```
CONF:VOLT:DC 10,1E-5
         │    │   └── resolution you want, in volts
         │    └────── range: the largest input you expect
         └─────────── function: DC volts
```

The range is not a suggestion. Ask for 3 V and you get the 10 V range, because
that is the smallest range the hardware actually has that will hold 3 V. Always
read a setting back rather than assuming it took.

In [ ]:
dmm.reset_input_buffer()

dmm.write(b"*RST\n")                     # known state: DC volts, autorange
dmm.write(b"CONF:VOLT:DC 10,1E-5\n")     # 10 V range, 10 uV resolution

# CONF? reports what the instrument actually settled on.
dmm.write(b"CONF?\n")
print("configured as:", dmm.read_until(b"\n").decode().strip())

# Ask for a range it does not have, and watch it round up.
dmm.write(b"VOLT:DC:RANG 3\n")
dmm.write(b"VOLT:DC:RANG?\n")
print("asked for 3 V, got:", dmm.read_until(b"\n").decode().strip())

dmm.write(b"VOLT:DC:RANG:AUTO ON\n")     # back to autoranging

In [ ]:
dmm.write(b"READ?\n")
reply = dmm.read_until(b"\n").decode().strip()

print("reply as text :", repr(reply))

volts = float(reply)
print("reply as float:", volts)
print(f"measured      : {volts:.6f} V")

`+5.00019000E+00` is the instrument's number format: explicit sign, nine significant
figures, two-digit exponent. Python's `float()` swallows it without complaint,
which is why you should never be tempted to parse it by hand.

Now turn the knob on the front panel — drag the **Level** slider in the signal
source frame — and run the cell above again. The number follows. That panel is
standing in for whatever you would have clipped to the test leads on a real bench.

---
## 3 · A hundred readings: mean, standard deviation, and a plot

One reading tells you almost nothing. A hundred tells you what the noise floor is,
and whether the thing you are measuring is drifting.

This loop is deliberately the naive one: send, wait, read, repeat. Watch the
traffic log while it runs and you can see your own command latency accumulating.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 100

dmm.reset_input_buffer()
dmm.write(b"CONF:VOLT:DC 10\n")
dmm.write(b"VOLT:DC:NPLC 1\n")           # one power-line cycle per reading

readings = []
stamps = []

t0 = time.perf_counter()
for _ in range(N):
    dmm.write(b"READ?\n")
    readings.append(float(dmm.read_until(b"\n")))
    stamps.append(time.perf_counter() - t0)
elapsed = time.perf_counter() - t0

mean = np.mean(readings)
sd = np.std(readings)

print(f"{N} readings in {elapsed:.2f} s  ->  {N / elapsed:.1f} readings/s")
print(f"mean    {mean:.6f} V")
print(f"std dev {sd * 1e6:.1f} uV   ({sd / abs(mean) * 1e6:.0f} ppm of reading)")
print(f"min/max {min(readings):.6f} / {max(readings):.6f} V")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4),
                               gridspec_kw={"width_ratios": [2, 1]})

ax1.plot(stamps, readings, ".-", ms=4, lw=0.8, color="#1f77b4")
ax1.axhline(mean, color="#d62728", lw=1.2, label=f"mean {mean:.6f} V")
ax1.fill_between([stamps[0], stamps[-1]], mean - sd, mean + sd,
                 color="#d62728", alpha=0.15, label=f"±1σ = ±{sd * 1e6:.0f} µV")
ax1.set_xlabel("time since first reading (s)")
ax1.set_ylabel("volts")
ax1.set_title(f"{N} readings of DC volts")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(alpha=0.3)

ax2.hist(readings, bins=20, orientation="horizontal", color="#1f77b4",
         edgecolor="white")
ax2.axhline(mean, color="#d62728", lw=1.2)
ax2.set_xlabel("count")
ax2.set_title("distribution")
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

---
## 4 · Tidy up

Check the error queue before you go. A clean run ends with `+0,"No error"`;
anything else means the instrument quietly rejected something you sent and you
never noticed.

In [ ]:
dmm.reset_input_buffer()

while True:
    dmm.write(b"SYST:ERR?\n")
    err = dmm.read_until(b"\n").decode().strip()
    print("  ", err)
    if err.startswith("+0"):
        break

dmm.write(b"*RST\n")
dmm.close()
print("port closed:", not dmm.is_open)